# 4.1 Data Ingestion

In [ ]:
from notebookutils import mssparkutils

LAKEHOUSE = "msba305"   # replace with your actual Lakehouse name
BASE_PATH = "Files/project_data"  # Relative path - Fabric requirement

PATHS = {
    "raw_weather":    f"{BASE_PATH}/raw/weather",
    "raw_311":        f"{BASE_PATH}/raw/311",
    "raw_traffic":    f"{BASE_PATH}/raw/traffic",
    "bronze_weather": f"{BASE_PATH}/bronze/weather",
    "bronze_311":     f"{BASE_PATH}/bronze/311",
    "bronze_traffic": f"{BASE_PATH}/bronze/traffic",
    "silver":         f"{BASE_PATH}/silver",
    "gold":           f"{BASE_PATH}/gold",
}

# Create directories using Fabric's file system API with relative paths
for name, path in PATHS.items():
    try:
        mssparkutils.fs.mkdirs(path)
        print(f" {name}: {path}")
    except Exception as e:
        print(f"{name} creation issue: {e}")

print(f"\nAll folders ready under {BASE_PATH}")
# testing if i can modify 

StatementMeta(, , -1, Waiting, , Waiting, True)

# Weather API -JSON
The structure is:
Top-level metadata (lat, lon, timezone, elevation, units)
hourly → a dictionary where each key is a field, and the value is a list of 8,760 values (one per hour × 365 days)
All lists are parallel — index 0 of time matches index 0 of temperature_2m, etc.

In [5]:
import requests
import json
import time
import datetime
from notebookutils import mssparkutils

def fetch_with_retry(url, params, max_retries=3, backoff=5):
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            print(f" Success on attempt {attempt}")
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f" Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(backoff)
    raise Exception("All retries failed.")

today = datetime.date.today()
start_date = today - datetime.timedelta(days=7)  # last 7 days

params = {
    "latitude":   40.71,
    "longitude":  -74.01,
    "start_date": "2026-01-01",
    "end_date":   start_date.isoformat(),
    "hourly":     "temperature_2m,precipitation,wind_speed_10m,weathercode"
}

raw_weather = fetch_with_retry(
    "https://archive-api.open-meteo.com/v1/archive", params
)

# Save raw JSON to Fabric Lakehouse
weather_raw_path = f"{PATHS['raw_weather']}/open_meteo_nyc_{today.isoformat()}.json"

try:
    # Write JSON string directly to OneLake using mssparkutils
    json_content = json.dumps(raw_weather, indent=2)
    mssparkutils.fs.put(weather_raw_path, json_content, overwrite=True)
    
    print(f" Weather raw JSON saved: {weather_raw_path}")
    print(f"   Records: {len(raw_weather.get('hourly', {}).get('time', []))} hours")
    
except Exception as e:
    print(f"Failed to save file: {e}")

StatementMeta(, 5a022f3e-fa28-4090-8697-32cae812b748, 7, Finished, Available, Finished, False)

 Success on attempt 1
 Weather raw JSON saved: Files/project_data/raw/weather/open_meteo_nyc_2026-04-16.json
   Records: 2376 hours


## NYC 311 Service Requests — Ingestion
This dataset is massive (10M rows), so the strategy is:

Download a manageable sample via the Socrata API (the system it runs on)
Save to raw layer

In [6]:
def fetch_311_data(limit=100000, year=2023):
    BASE_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
    all_records = []
    offset = 0
    batch_size = 50000

    print(f"📥 Fetching NYC 311 data for {year}...")
    while len(all_records) < limit:
        params = {
            "$limit":  batch_size,
            "$offset": offset,
            "$where":   f"created_date >= '2026-01-01T00:00:00' AND created_date < '{today}T00:00:00'",
            "$order":  "created_date ASC"
        }
        for attempt in range(1, 4):
            try:
                response = requests.get(BASE_URL, params=params, timeout=60)
                response.raise_for_status()
                batch = response.json()
                break
            except requests.exceptions.RequestException as e:
                print(f"   Attempt {attempt} failed: {e}")
                if attempt < 3:
                    time.sleep(5)
                else:
                    raise Exception("All retries failed.")

        if not batch:
            break

        all_records.extend(batch)
        offset += batch_size
        print(f"   Fetched {len(all_records):,} records so far...")

        if len(batch) < batch_size:
            break

    print(f"\n Total: {len(all_records):,} records")
    return all_records

raw_311 = fetch_311_data(limit=100000, year=2023)

# Save raw JSON to Fabric Lakehouse
path_311 = f"{PATHS['raw_311']}/nyc_311_{today.isoformat()}.json"

try:
    # Write JSON content directly to OneLake using mssparkutils
    json_content = json.dumps(raw_311, indent=2)
    mssparkutils.fs.put(path_311, json_content, overwrite=True)
    
    file_size_mb = len(json_content.encode('utf-8')) / 1024 / 1024
    print(f" 311 raw JSON saved: {path_311}")
    print(f"   File size: {file_size_mb:.1f} MB")
    print(f"   Record count: {len(raw_311):,}")
    
except Exception as e:
    print(f" Failed to save file: {e}")

StatementMeta(, 5a022f3e-fa28-4090-8697-32cae812b748, 8, Submitted, Running, Running, True)

📥 Fetching NYC 311 data for 2023...
   Fetched 50,000 records so far...


# Automated Traffic Volume Counts — Ingestion
This dataset is massive (1.88M rows), so the strategy is:

Download a manageable sample via the Socrata API (the system it runs on)
Save to raw layer

In [ ]:
def fetch_traffic_data():
    url = "https://data.cityofnewyork.us/api/views/7ym2-wayt/rows.csv?accessType=DOWNLOAD"
    path = f"{PATHS['raw_traffic']}/nyc_traffic_counts_{today.isoformat()}.csv"

    print("📥 Downloading NYC Traffic Volume Counts...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()

    # Save CSV directly to OneLake
    mssparkutils.fs.put(path, response.text, overwrite=True)

    size_mb = len(response.text.encode('utf-8')) / 1024 / 1024
    print(f" Traffic CSV saved: {path}")
    print(f"   File size: {size_mb:.1f} MB")
    return path

traffic_raw_path = fetch_traffic_data()

StatementMeta(, , -1, Waiting, , Waiting, True)

# Parsing all 3 sources into the bronze layer: structured and uncleaned. 

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json

def ingest_weather_to_bronze(raw_path, bronze_path):
    """
    Reads raw Open-Meteo JSON with proper schema handling
    """
    print("📥 Reading raw weather JSON with Spark...")
    
    # Define the exact schema for Open-Meteo response
    schema = StructType([
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("generationtime_ms", DoubleType(), True),
        StructField("utc_offset_seconds", IntegerType(), True),
        StructField("timezone", StringType(), True),
        StructField("timezone_abbreviation", StringType(), True),
        StructField("elevation", DoubleType(), True),
        StructField("hourly", StructType([
            StructField("time", ArrayType(StringType()), True),
            StructField("temperature_2m", ArrayType(DoubleType()), True),
            StructField("precipitation", ArrayType(DoubleType()), True),
            StructField("wind_speed_10m", ArrayType(DoubleType()), True),
            StructField("weathercode", ArrayType(IntegerType()), True),
        ]), True)
    ])
    
    # Read JSON with schema
    df_raw = spark.read.schema(schema).json(raw_path)
    
    # Extract latitude/longitude before exploding
    latitude = df_raw.select(F.col("latitude")).collect()[0][0]
    longitude = df_raw.select(F.col("longitude")).collect()[0][0]
    
    # Explode the hourly arrays into rows
    df = df_raw.select(
        F.explode(
            F.arrays_zip(
                F.col("hourly.time"),
                F.col("hourly.temperature_2m"),
                F.col("hourly.precipitation"),
                F.col("hourly.wind_speed_10m"),
                F.col("hourly.weathercode")
            )
        ).alias("hourly_data")
    ).select(
        F.col("hourly_data.time").alias("timestamp"),
        F.col("hourly_data.temperature_2m").alias("temperature_2m"),
        F.col("hourly_data.precipitation").alias("precipitation"),
        F.col("hourly_data.wind_speed_10m").alias("wind_speed_10m"),
        F.col("hourly_data.weathercode").alias("weathercode")
    )
    
    # Add metadata columns
    df = df.withColumn("timestamp", F.to_timestamp("timestamp")) \
           .withColumn("source", F.lit("open-meteo-archive")) \
           .withColumn("latitude", F.lit(latitude)) \
           .withColumn("longitude", F.lit(longitude)) \
           .withColumn("ingested_at", F.current_timestamp())
    
    # Write as Delta table
    df.write.format("delta").mode("overwrite").save(bronze_path)
    
    print(f" Weather bronze Delta table written: {bronze_path}")
    print(f"   Rows: {df.count():,}")
    df.printSchema()
    return df

df_weather_bronze = ingest_weather_to_bronze(
    raw_path    = f"{PATHS['raw_weather']}/open_meteo_nyc_{today.isoformat()}.json",
    bronze_path = PATHS["bronze_weather"]
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

def ingest_311_to_bronze(raw_path, bronze_path):
    print("📥 Reading raw 311 JSON with Spark...")
    df = spark.read.json(raw_path, multiLine=True)

    print(f"   Raw columns available: {df.columns}")

    # Keep only needed columns
    cols_we_need = [
        "unique_key", "created_date", "closed_date",
        "complaint_type", "descriptor", "borough",
        "city", "latitude", "longitude",
        "status", "agency", "agency_name"
    ]
    cols_available = [c for c in cols_we_need if c in df.columns]
    df = df.select(cols_available)

    # Cast types + add metadata
    df = df.withColumn("created_date", F.to_timestamp("created_date")) \
           .withColumn("closed_date",  F.to_timestamp("closed_date")) \
           .withColumn("latitude",     F.col("latitude").cast(DoubleType())) \
           .withColumn("longitude",    F.col("longitude").cast(DoubleType())) \
           .withColumn("source",       F.lit("nyc-open-data-311")) \
           .withColumn("ingested_at",  F.current_timestamp())

    df.write.format("delta").mode("overwrite").save(bronze_path)

    print(f" 311 bronze Delta table written: {bronze_path}")
    print(f"   Rows: {df.count():,}")
    df.printSchema()
    return df

#  Call with correct file path (matches your fetch naming)
df_311_bronze = ingest_311_to_bronze(
    raw_path    = f"{PATHS['raw_311']}/nyc_311_{today.isoformat()}.json",
    bronze_path = PATHS["bronze_311"]
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
def ingest_traffic_to_bronze(raw_path, bronze_path):
    """
    Reads raw traffic CSV using PySpark (handles 223MB efficiently),
    reconstructs a proper timestamp from split date columns,
    and writes a Delta table to the bronze layer.
    """
    print("📥 Reading raw traffic CSV...")

    df = spark.read.csv(raw_path, header=True, inferSchema=True)

    print(f"   Raw columns: {df.columns}")

    # Select relevant columns
    cols_we_need = [
        "RequestID", "Boro", "Yr", "M", "D", "HH", "MM",
        "Vol", "SegmentID", "street", "fromSt", "toSt", "Direction"
    ]
    cols_available = [c for c in cols_we_need if c in df.columns]
    df = df.select(cols_available)

    # Reconstruct timestamp from split date/time columns
    df = df.withColumn(
        "timestamp",
        F.to_timestamp(
            F.concat_ws("-",
                F.col("Yr").cast(StringType()),
                F.lpad(F.col("M").cast(StringType()),  2, "0"),
                F.lpad(F.col("D").cast(StringType()),  2, "0"),
                F.lpad(F.col("HH").cast(StringType()), 2, "0"),
                F.lpad(F.col("MM").cast(StringType()), 2, "0")
            ),
            "yyyy-MM-dd-HH-mm"
        )
    )

    # Cast volume
    df = df.withColumn("Vol", F.col("Vol").cast(IntegerType())) \
           .withColumn("source",      F.lit("nyc-open-data-traffic")) \
           .withColumn("ingested_at", F.current_timestamp())

    df.write.format("delta").mode("overwrite").save(bronze_path)

    print(f" Traffic bronze Delta table written: {bronze_path}")
    print(f"   Rows: {df.count():,}")
    df.printSchema()
    return df

df_traffic_bronze = ingest_traffic_to_bronze(
    raw_path    = f"{PATHS['raw_traffic']}/nyc_traffic_counts_{today.isoformat()}.csv",
    bronze_path = PATHS["bronze_traffic"]
)

StatementMeta(, , -1, Waiting, , Waiting, True)